Vogliamo costruire un dataset Amazon-like multi-categoria, intersecando:
Amazon Reviews 2023 che contiene metadati + rating/recensioni
Amazon question/answer storico McAuley

Questo notebook costruisce il dataset test_dataset2 intersecando più categorie:
- QA storico McAuley
- metadati Amazon Reviews 2023
- rating/recensioni Amazon Reviews 2023

Categorie usate:
- Electronics
- Cell Phones and Accessories
- Video Games
- Software
- Office Products
- Appliances

Output finale:
- items.parquet
- qa.parquet
- ratings.parquet
- category_summary.csv
- summary.json

In [1]:
# Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path #per gestire meglio percorsi di file e cartelle
import ast #servirà più avanti per leggere file del QA storico
import gzip #leggere e scrivere file compressi gz
import json #lavorare con json
from typing import Iterable, Optional #tipi
import pandas as pd #pandas per creare dataframe
import pyarrow as pa #per definire schemi tabellari
import pyarrow.parquet as pq    #per leggere metadata dei file .parquet
import requests #per scaricare da URL
from tqdm.auto import tqdm #barra di progresso
import datasets #pacchetto datasets
from datasets import load_dataset #per caricare dataset

datasets.logging.set_verbosity_error() #riduce i messaggi stampati da datasets

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configurazione
DATASET_NAME="test_dataset3" #nome tecnico usato per cartelle di output
DATASET_LABEL="AmazonLike_v2" #nome leggibile del dataset

CATEGORIES=[
    {"label":"Electronics","hf_category":"Electronics","qa_slug":"Electronics","group":"tech"},
    {"label":"Video Games","hf_category":"Video_Games","qa_slug":"Video_Games","group":"tech"},
    {"label":"Software","hf_category":"Software","qa_slug":"Software","group":"tech"},
    {"label":"Office Products","hf_category":"Office_Products","qa_slug":"Office_Products","group":"office"},
    {"label":"Appliances","hf_category":"Appliances","qa_slug":"Appliances","group":"home"},
    {"label":"Musical Instruments","hf_category":"Musical_Instruments","qa_slug":"Musical_Instruments","group":"music"},
    {"label":"Arts Crafts and Sewing","hf_category":"Arts_Crafts_and_Sewing","qa_slug":"Arts_Crafts_and_Sewing","group":"crafts"},

    {"label":"Automotive","hf_category":"Automotive","qa_slug":"Automotive","group":"automotive"},
    {"label":"Baby Products","hf_category":"Baby_Products","qa_slug":"Baby","group":"baby"},
    {"label":"Grocery and Gourmet Food","hf_category":"Grocery_and_Gourmet_Food","qa_slug":"Grocery_and_Gourmet_Food","group":"grocery"},
    {"label":"Home and Kitchen","hf_category":"Home_and_Kitchen","qa_slug":"Home_and_Kitchen","group":"home"},
    {"label":"Patio Lawn and Garden","hf_category":"Patio_Lawn_and_Garden","qa_slug":"Patio_Lawn_and_Garden","group":"garden"},
    {"label":"Pet Supplies","hf_category":"Pet_Supplies","qa_slug":"Pet_Supplies","group":"pets"},
    {"label":"Sports and Outdoors","hf_category":"Sports_and_Outdoors","qa_slug":"Sports_and_Outdoors","group":"sports"},
    {"label":"Tools and Home Improvement","hf_category":"Tools_and_Home_Improvement","qa_slug":"Tools_and_Home_Improvement","group":"tools"},
    {"label":"Toys and Games","hf_category":"Toys_and_Games","qa_slug":"Toys_and_Games","group":"toys"},
]
#label è il nome della categoria che usiamo noi
#hfcategory è il nome usato su amazon reviews
#qaslug è il nome del file qa di quella categoria
#group macro categoria creata da me

QA_BASE_URL="https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa"
MAX_ITEMS_BY_CATEGORY={
    "Electronics":10000,
    "Video Games":600,
    "Software":600,
    "Office Products":1500,
    "Appliances":600,
    "Musical Instruments":1200,
    "Arts Crafts and Sewing":1000,

    "Automotive":500,
    "Baby Products":500,
    "Grocery and Gourmet Food":500,
    "Home and Kitchen":700,
    "Patio Lawn and Garden":500,
    "Pet Supplies":500,
    "Sports and Outdoors":700,
    "Tools and Home Improvement":700,
    "Toys and Games":700,
}
MAX_METADATA_SCAN_BY_CATEGORY={
    "Electronics":400000,
    "Video Games":150000,
    "Software":100000,
    "Office Products":450000,
    "Appliances":120000,
    "Musical Instruments":180000,
    "Arts Crafts and Sewing":550000,

    "Automotive":150000,
    "Baby Products":150000,
    "Grocery and Gourmet Food":150000,
    "Home and Kitchen":200000,
    "Patio Lawn and Garden":150000,
    "Pet Supplies":150000,
    "Sports and Outdoors":200000,
    "Tools and Home Improvement":200000,
    "Toys and Games":200000,
}
MAX_REVIEWS_SCAN_BY_CATEGORY={
    "Electronics":1200000,
    "Video Games":400000,
    "Software":250000,
    "Office Products":600000,
    "Appliances":250000,
    "Musical Instruments":400000,
    "Arts Crafts and Sewing":500000,

    "Automotive":250000,
    "Baby Products":250000,
    "Grocery and Gourmet Food":250000,
    "Home and Kitchen":300000,
    "Patio Lawn and Garden":250000,
    "Pet Supplies":250000,
    "Sports and Outdoors":300000,
    "Tools and Home Improvement":300000,
    "Toys and Games":300000,
}

MAX_RATINGS_SAVED_BY_CATEGORY={
    "Electronics":350000,
    "Video Games":100000,
    "Software":70000,
    "Office Products":140000,
    "Appliances":70000,
    "Musical Instruments":100000,
    "Arts Crafts and Sewing":100000,

    "Automotive":50000,
    "Baby Products":50000,
    "Grocery and Gourmet Food":50000,
    "Home and Kitchen":60000,
    "Patio Lawn and Garden":50000,
    "Pet Supplies":50000,
    "Sports and Outdoors":60000,
    "Tools and Home Improvement":60000,
    "Toys and Games":60000,
}
SAVE_TEXT_OFREVIEW=True
FORCE_REBUILD_RATINGS=False #se True ricrea ratings.parquet anche se esiste già

In [4]:
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
RAW_DIR=PROJECT_ROOT/"data"/"raw"
PROCESSED_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_intersection"
QA_RAW_DIR=RAW_DIR/"qa"

ITEMS_OUT=PROCESSED_DIR/"items.parquet" #percorso finale dei prodotti
QA_OUT=PROCESSED_DIR/"qa.parquet" #percorso file domande/risposte
RATINGS_OUT=PROCESSED_DIR/"ratings.parquet" #percorso rating
CATEGORY_SUMMARY_PATH=PROCESSED_DIR/"category_summary.csv" #riepilogo per categoria
SUMMARY_PATH=PROCESSED_DIR/"summary.json" #riepilogo generale

RAW_DIR.mkdir(parents=True,exist_ok=True) #crea data/raw
QA_RAW_DIR.mkdir(parents=True,exist_ok=True) #crea data/raw/qa
PROCESSED_DIR.mkdir(parents=True,exist_ok=True) #crea cartella output dataset2

print("Project root:",PROJECT_ROOT)
print("Output:",PROCESSED_DIR)
print("Categorie selezionate:")
for category in CATEGORIES:
    print("-",category["label"],"| hf=",category["hf_category"],"| qa=",category["qa_slug"])


Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset3_intersection
Categorie selezionate:
- Electronics | hf= Electronics | qa= Electronics
- Video Games | hf= Video_Games | qa= Video_Games
- Software | hf= Software | qa= Software
- Office Products | hf= Office_Products | qa= Office_Products
- Appliances | hf= Appliances | qa= Appliances
- Musical Instruments | hf= Musical_Instruments | qa= Musical_Instruments
- Arts Crafts and Sewing | hf= Arts_Crafts_and_Sewing | qa= Arts_Crafts_and_Sewing
- Automotive | hf= Automotive | qa= Automotive
- Baby Products | hf= Baby_Products | qa= Baby
- Grocery and Gourmet Food | hf= Grocery_and_Gourmet_Food | qa= Grocery_and_Gourmet_Food
- Home and Kitchen | hf= Home_and_Kitchen | qa= Home_and_Kitchen
- Patio Lawn and Garden | hf= Patio_Lawn_and_Garden | qa= Patio_Lawn_and_Garden
- Pet Supplies | hf= Pet_Supplies | qa= Pet_Supplies
- Sports and Outdoors | hf= Sports_and_Outdoors |

In [5]:
def download_file(url:str,output_path:Path,force:bool=False)->Path: #scarica da url in output_path
    if output_path.exists() and not force: #non riscarica
        print(f"Già presente: {output_path}")
        return output_path
    output_path.parent.mkdir(parents=True,exist_ok=True) #se non esiste cartella di destinazione la crea

    with requests.get(url,stream=True,timeout=60)as r:
        r.raise_for_status()
        total=int(r.headers.get("content-length",0)) #dimensione
        with open(output_path,"wb") as f,tqdm( #apre file in scrittura binaria, perchè scarichi .gz e apre barra tqdm
            total=total,
            unit="B", #in byte
            unit_scale=True,
            desc=output_path.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return output_path

In [6]:
#Convertiamo QA storico da JSONL gzip a JSON e aggiungiamo parent_asin per
#agganciarlo al dataset amazon 2023
#per fare questo l'asin storico lo usiamo come parent_asin

def iter_historical_qa(path:Path)->Iterable[dict]: #legge qa storico, restituisce dizionari python
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as f: #apre il file .gz
        for line in f:
            line=line.strip() #rimuove spazi e newline
            if line:
                yield ast.literal_eval(line) #converte riga da testo a dizionario python

#dizionari per info per ogni categoria
qa_info_by_category={}
category_summary_rows=[]

for category in CATEGORIES: #cicliamo ogni categoria
    qa_slug=category["qa_slug"]
    qa_url=f"{QA_BASE_URL}/qa_{qa_slug}.json.gz"
    #file originale scaricato e file convertito
    qa_raw_path=QA_RAW_DIR/f"qa_{qa_slug}.json.gz"
    qa_strict_path=QA_RAW_DIR/f"qa_{qa_slug}.strict.jsonl.gz"

    download_file(qa_url,qa_raw_path) #scarica da internet

    qa_asins=set()
    qa_rows=0

    with gzip.open(qa_strict_path,"wt",encoding="utf-8") as out: #apre il file in scrittura, .jsonl.gz
        for qa_id,row in enumerate(tqdm(iter_historical_qa(qa_raw_path),desc=f"Conversione QA {category['label']}")):
            asin=row.get("asin") #prende valore dal campo asin
            row["qa_id"]=qa_id #aggiunge id numerico alla riga qa nella categoria
            row["parent_asin"]=asin #crea il campo parent_asin e ci copia l'asin
            row["source_category"]=category["label"] #categoria
            row["hf_category"]=category["hf_category"] #categoria huggingface
            row["qa_slug"]=category["qa_slug"] #file qa da cui arriva
            row["category_group"]=category["group"] #macrogruppo

            if asin:
                qa_asins.add(asin)

            out.write(json.dumps(row,ensure_ascii=False)+"\n") #scrive la riga nel nuovo file jsonl
            qa_rows+=1

    qa_info_by_category[category["label"]]={ #info qa per categoria
        "qa_raw_path":qa_raw_path,
        "qa_strict_path":qa_strict_path,
        "qa_asins":qa_asins,
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    }
    category_summary_rows.append({ #riepilogo senza asin, più leggero
        "source_category":category["label"],
        "hf_category":category["hf_category"],
        "qa_slug":category["qa_slug"],
        "category_group":category["group"],
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    })

    print(category["label"],"| QA rows:",qa_rows,"| prodotti QA:",len(qa_asins),"| file strict:",qa_strict_path)

Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.json.gz


Conversione QA Electronics: 314263it [00:12, 25588.14it/s]


Electronics | QA rows: 314263 | prodotti QA: 39371 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Video_Games.json.gz


Conversione QA Video Games: 13307it [00:00, 29205.74it/s]


Video Games | QA rows: 13307 | prodotti QA: 1188 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Video_Games.strict.jsonl.gz
Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Software.json.gz


Conversione QA Software: 10636it [00:00, 23248.63it/s]


... (23 outputs left out)

qa_Tools_and_Home_Improvement.json.gz: 100%|██████████| 11.0M/11.0M [00:05<00:00, 1.91MB/s]
Conversione QA Tools and Home Improvement: 101088it [00:04, 24817.55it/s]


Tools and Home Improvement | QA rows: 101088 | prodotti QA: 13397 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Tools_and_Home_Improvement.strict.jsonl.gz


qa_Toys_and_Games.json.gz: 100%|██████████| 4.81M/4.81M [00:02<00:00, 1.77MB/s]
Conversione QA Toys and Games: 51486it [00:01, 25923.54it/s]

Toys and Games | QA rows: 51486 | prodotti QA: 7393 | file strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Toys_and_Games.strict.jsonl.gz


In [7]:
#prepariamo funzioni comuni e schema dei prodotti
#ogni riga salvata mantiene anche source_category per analisi multi-categoria

def image_url(images)->Optional[str]: #prova a estrarre immagine da campo images
    if isinstance(images,dict):
        for key in ("hi_res","large","thumb"):
            values=images.get(key) or []
            for value in values:
                if value:
                    return value
    if isinstance(images,list):
        for img in images:
            if isinstance(img,dict):
                for key in ("large_image_url","medium_image_url","small_image_url"):
                    value=img.get(key)
                    if value:
                        return value
    return None

def tojson_string(value)->str: #converte valore in stringa json
    if value is None:
        return "null"
    return json.dumps(value,ensure_ascii=False)

def tofloat_or_none(value): #converte valore in numero decimale
    try:
        if value in (None,"","None"):
            return None
        return float(value)
    except Exception:
        return None

def toint_or_none(value): #converte in intero
    try:
        if value in (None,"","None"):
            return None
        return int(value)
    except Exception:
        return None

def get_category(row:dict,default_category:str)->Optional[str]:
    main_category=row.get("main_category")
    if main_category not in (None,"","None"):
        return main_category

    categories=row.get("categories")
    if isinstance(categories,list) and len(categories)>0:
        return categories[0]

    return default_category

ITEM_SCHEMA=pa.schema([ #schema del file items.parquet
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("title",pa.string()),
    ("category",pa.string()),
    ("avg_rating",pa.float64()),
    ("num_ratings",pa.int64()),
    ("price",pa.float64()),
    ("store",pa.string()),
    ("features_json",pa.string()),
    ("description_json",pa.string()),
    ("categories_json",pa.string()),
    ("details_json",pa.string()),
    ("image_url",pa.string()),
])

#scrive righe in un file parquet, rows=listadiz, path=dove salvare, schema=schema PyArrow, writer=oggetto che gestisce la scrittura Parquet
def writerows_in_parquet(rows:list[dict],path:Path,schema:pa.schema,writer:Optional[pq.ParquetWriter]):
    if not rows:
        return writer

    df=pd.DataFrame(rows) #converte lista di dizionari in un dataframe pandas
    table=pa.Table.from_pandas(df,schema=schema,preserve_index=False) #converte dataframe in tabella pyarrow

    if writer is None:
        path.parent.mkdir(parents=True,exist_ok=True) #crea cartella di output
        writer=pq.ParquetWriter(path,schema=schema,compression="zstd") #crea un writer Parquet,compresso
    writer.write_table(table)
    return writer

In [8]:
#carichiamo metadati amazon reviews 2023 da Hugging Face per ogni categoria
#teniamo solo prodotti presenti anche nel QA e deduplichiamo parent_asin fra categorie
#creaiamo intes.parquet cioè tab prodotti finali
#leggiamo direttamente i file JSONL riga per riga, così possiamo fermarci prima

HF_DATASET_BASE_URL="https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main"

def iter_hf_jsonl(relative_path): #legge un file jsonl da Hugging Face senza scaricarlo tutto prima
    url=f"{HF_DATASET_BASE_URL}/{relative_path}"
    with requests.get(url,stream=True,timeout=(30,300)) as response:
        response.raise_for_status()
        for line in response.iter_lines(chunk_size=1024*1024):
            if line:
                yield json.loads(line)

if ITEMS_OUT.exists(): #se items.parquet esiste già lo elimina
    ITEMS_OUT.unlink()

writer=None
buffer=[]
matched_items_total=0
global_items_seen=set()
items_id_by_category={}

for category in CATEGORIES: #cicla ogni categoria
    label=category["label"]
    hf_category=category["hf_category"]
    qa_asins=qa_info_by_category[label]["qa_asins"]

    meta_path=f"raw/meta_categories/meta_{hf_category}.jsonl"
    meta_ds=iter_hf_jsonl(meta_path) #risultato salvato in meta_ds

    matched_items=0
    duplicates_skipped=0
    category_items=set()
    metadata_rows_scanned=0
    
    max_items=None
    if isinstance(MAX_ITEMS_BY_CATEGORY,dict):
        max_items=MAX_ITEMS_BY_CATEGORY.get(label)

    max_metadata_scan=None
    if isinstance(MAX_METADATA_SCAN_BY_CATEGORY,dict):
        max_metadata_scan=MAX_METADATA_SCAN_BY_CATEGORY.get(label)
    
    for row in tqdm(meta_ds,desc=f"Filtro metadati {label}"): #legge una riga prodotto alla  volta dal dataset huggingface
        if max_metadata_scan is not None and metadata_rows_scanned>=max_metadata_scan:
            break
        metadata_rows_scanned+=1

        parent_asin=row.get("parent_asin")
        if parent_asin not in qa_asins: #salta items non in qa storico
            continue
        if parent_asin in global_items_seen: #salta prodotto se compare in più categorie, senza duplicati
            duplicates_skipped+=1
            continue

        buffer.append({ #ggiunge prodotto al buffer
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":hf_category,
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "title":row.get("title"),
            "category":get_category(row,label),
            "avg_rating":tofloat_or_none(row.get("average_rating")),
            "num_ratings":toint_or_none(row.get("rating_number")),
            "price":tofloat_or_none(row.get("price")),
            "store":row.get("store"),
            "features_json":tojson_string(row.get("features")),
            "description_json":tojson_string(row.get("description")),
            "categories_json":tojson_string(row.get("categories")),
            "details_json":tojson_string(row.get("details")),
            "image_url":image_url(row.get("images")),
        })

        matched_items+=1
        matched_items_total+=1
        category_items.add(parent_asin)
        global_items_seen.add(parent_asin)

        if max_items is not None and matched_items>=max_items:
            break

        if len(buffer)>=50_000: #ogni 50k righe svuota il buffer e scrive nel parquet
            writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)
            buffer=[]

    items_id_by_category[label]=category_items #salva insieme dei prodotti di quella categoria
    for summary_row in category_summary_rows: #aggiorniamo riepilogo per categoria, con dati 
        if summary_row["source_category"]==label:
            summary_row["metadata_rows"]=int(metadata_rows_scanned) #prodotti metadati della categoria
            summary_row["items_qa_metadata"]=int(matched_items) #prodotti salvati perchè sia in qa sia metadata
            summary_row["duplicate_items_skipped"]=int(duplicates_skipped) #prodotti skipapti perchè gia presi in altre categorie
            break

    print(label,"| metadati scansionati:",metadata_rows_scanned,"| prodotti QA ∩ metadata salvati:",matched_items,"| duplicati saltati:",duplicates_skipped)

writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)#ultimo buffer rimasto
if writer is not None:
    writer.close()

items_df=pd.read_parquet(ITEMS_OUT)
items_id=set(items_df["parent_asin"])

print("Prodotti QA ∩ metadata 2023 totali:",len(items_id))
display(items_df.head())

Filtro metadati Electronics: 400000it [00:30, 13248.63it/s]


Electronics | metadati scansionati: 400000 | prodotti QA ∩ metadata salvati: 6142 | duplicati saltati: 0


Filtro metadati Video Games: 116963it [00:09, 12771.17it/s]


Video Games | metadati scansionati: 116964 | prodotti QA ∩ metadata salvati: 600 | duplicati saltati: 0


Filtro metadati Software: 85067it [00:06, 13919.60it/s]


Software | metadati scansionati: 85068 | prodotti QA ∩ metadata salvati: 600 | duplicati saltati: 0


... (23 outputs left out)

Tools and Home Improvement | metadati scansionati: 146161 | prodotti QA ∩ metadata salvati: 700 | duplicati saltati: 0


Filtro metadati Toys and Games: 178332it [00:12, 14056.57it/s]


Toys and Games | metadati scansionati: 178333 | prodotti QA ∩ metadata salvati: 700 | duplicati saltati: 0
Prodotti QA ∩ metadata 2023 totali: 16672


,parent_asin,source_category,hf_category,qa_slug,category_group,title,category,avg_rating,num_ratings,price,store,features_json,description_json,categories_json,details_json,image_url
0,B00CFGZAT8,Electronics,Electronics,Electronics,tech,Dock Audio Extender Adapter Converter Cable fo...,Portable Audio & Accessories,3.6,45,NaN,iteck,[],[],"[""Electronics"", ""Computers & Accessories"", ""Ta...","{""Brand Name"": ""iTeck"", ""Item Weight"": ""0.8 ou...",NaN
1,B00BGFTSGU,Electronics,Electronics,Electronics,tech,GGS Swivi HD DSLR LCD Universal Foldable Viewf...,Camera & Photo,3.7,52,149.00,Swivi,[],[],"[""Electronics"", ""Camera & Photo"", ""Accessories...","{""Product Dimensions"": ""3.8 x 4.3 x 6.9 inches...",NaN
2,B00AZOIG9I,Electronics,Electronics,Electronics,tech,BM Premium 2-Pack of NP-85 Batteries and Charg...,Camera & Photo,4.7,345,22.99,BM Premium,"[""NP85 Li-ion Battery for FujiFilm FinePix SL2...","[""Bring your digital camera back to life with ...","[""Electronics"", ""Camera & Photo"", ""Accessories...","{""Package Dimensions"": ""6.46 x 3.94 x 1.85 inc...",NaN
3,B00F361KLO,Electronics,Electronics,Electronics,tech,ZAGG InvisibleShield HD – EZ Apply Film Screen...,Computers,4.1,158,NaN,InvisibleShield,"[""100% Clear: Independent light transmission t...","[""Exceptionally clear, unbelievably thin, and ...","[""Electronics"", ""Computers & Accessories"", ""Ta...","{""Standing screen display size"": ""9.7 Inches"",...",NaN
4,B0029L7N76,Electronics,Electronics,Electronics,tech,Mobile Edge ECO Laptop Messenger (Eco-Friendly...,Computers,3.5,12,NaN,Mobile Edge,"[""Fits 17.3-Inch laptops. Laptop compartment d...","[""The 17.3-Inch ECO Messenger is the latest ad...","[""Electronics"", ""Computers & Accessories"", ""La...","{""Product Dimensions"": ""4.25 x 18 x 13.5 inche...",NaN


In [9]:
# ora filtriamo QA e teniamo solo domande/risposte dei prodotti presenti anche nei metadati 2023
QA_SCHEMA=pa.schema([ #schema del file qa.parquet
    ("qa_id",pa.int64()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("question",pa.string()),
    ("answer",pa.string()),
    ("questionType",pa.string()),
    ("answerType",pa.string()),
    ("answerTime",pa.string()),
    ("unixTime",pa.int64()),
])

def iter_jsonl_gz(path:Path)->Iterable[dict]: #legge file .jsonl.gz
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as f:
        for line in f:
            line=line.strip() #pulisce stringa
            if line:
                yield json.loads(line)

if QA_OUT.exists(): #se qa.parquet esiste già lo elimina
    QA_OUT.unlink()

writer=None
buffer=[]
matched_qa_total=0

for category in CATEGORIES:
    label=category["label"]
    category_items=items_id_by_category[label] #insieme prodotti della categoria presa
    qa_strict_path=qa_info_by_category[label]["qa_strict_path"] #prende file qa convertito della categoria
    matched_qa=0

    for row in tqdm(iter_jsonl_gz(qa_strict_path),desc=f"Filtro QA {label}"): #legge riga per riga il file qa, con barra di avanzamento tqdm
        parent_asin=row.get("parent_asin")
        if parent_asin not in category_items: #skippa qa di prodotti non presenti in items.parquet
            continue

        buffer.append({ #aggiunge la riga qa al buffer
            "qa_id":int(row.get("qa_id")),
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":category["hf_category"],
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "question":row.get("question"),
            "answer":row.get("answer"),
            "questionType":row.get("questionType"),
            "answerType":row.get("answerType"),
            "answerTime":row.get("answerTime"),
            "unixTime":toint_or_none(row.get("unixTime")),
        })
        matched_qa+=1
        matched_qa_total+=1

        if len(buffer)>=100_000: #quando buffer è pieno, scrive nel qa.parquet
            writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
            buffer=[]

    for row_summary in category_summary_rows:
        if row_summary["source_category"]==label:
            row_summary["qa_rows_after_metadata"] = int(matched_qa)
            row_summary["qa_products_after_metadata"] = int(len(category_items))
            break
    print(label,"| QA filtrate:",matched_qa)

writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
if writer is not None:
    writer.close()

qa_df=pd.read_parquet(QA_OUT)
print("QA filtrate totali:",len(qa_df))
display(qa_df.head())

Filtro QA Electronics: 0it [00:00, ?it/s]

Filtro QA Electronics: 314263it [00:01, 234856.18it/s]


Electronics | QA filtrate: 48316


Filtro QA Video Games: 13307it [00:00, 227188.41it/s]


Video Games | QA filtrate: 6536


Filtro QA Software: 10636it [00:00, 209645.23it/s]


... (24 outputs left out)

Tools and Home Improvement | QA filtrate: 4939


Filtro QA Toys and Games: 51486it [00:00, 252900.49it/s]


Toys and Games | QA filtrate: 4936
QA filtrate totali: 128675


,qa_id,parent_asin,source_category,hf_category,qa_slug,category_group,question,answer,questionType,answerType,answerTime,unixTime
0,84,1400501741,Electronics,Electronics,Electronics,tech,Can I port my Kindle books to this platform?,You just need to download the Kindle app from ...,yes/no,?,"Apr 8, 2014",1.396940e+09
1,85,1400501741,Electronics,Electronics,Electronics,tech,Why is it 179.99? Everywhere else is 269.99......,Its 179 on the Barnes and Noble web site as we...,open-ended,NaN,"Feb 8, 2014",1.391846e+09
2,86,1400501741,Electronics,Electronics,Electronics,tech,Can you use Word2010 and OneNote2010 on the No...,"Word and OneNote have Android versions, howeve...",open-ended,NaN,"Jul 23, 2013",1.374563e+09
3,87,1400501741,Electronics,Electronics,Electronics,tech,Can I download the Kindle app for this?,Yes you can. I have it installed,yes/no,Y,"Mar 11, 2014",1.394521e+09
4,88,1400501741,Electronics,Electronics,Electronics,tech,Which version of Android does this run?,Originally from factory is shipped with Androi...,open-ended,NaN,"Nov 18, 2013",1.384762e+09


In [10]:
#scansioniamo reviews 2023 per tutte le categorie e salviamo solo interazioni dei prodotti che abbiamo già
#se ratings.parquet esiste già e FORCE_REBUILD_RATINGS è False, non lo ricreiamo
#creiamo quindi ratings.parquet
rating_fields=[ #lista di colonne per file di rating
    ("user_id",pa.string()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("asin",pa.string()),
    ("rating",pa.float64()),
    ("timestamp",pa.int64()),
    ("verified_purchase",pa.bool_()),
]
if SAVE_TEXT_OFREVIEW: #se piu avanti vogliamo usare recensioni per sistema qa o per altri sistemi recomendation
    rating_fields.extend([
        ("review_title",pa.string()),
        ("review_text",pa.string()),
    ])
RATING_SCHEMA=pa.schema(rating_fields)

if RATINGS_OUT.is_file() and RATINGS_OUT.stat().st_size>0 and not FORCE_REBUILD_RATINGS: #se ratingsout esiste, non è vuoto e forcerebuild è false
    n_rows=pq.ParquetFile(RATINGS_OUT).metadata.num_rows
    print(f"ratings.parquet già presente: {RATINGS_OUT}")
    print(f"Righe già presenti: {n_rows:,}")
    print("Saltiamo il filtraggio dei rating.")
else: #sennò elimina quello vecchio se presente, e lo ricreiamo
    if RATINGS_OUT.exists():
        RATINGS_OUT.unlink()

    writer=None
    buffer=[]
    matched_ratings_total=0
    scanned_total=0

    for category in CATEGORIES: #cicla categorie
        label=category["label"]
        hf_category=category["hf_category"]
        category_items=items_id_by_category[label]
        max_scan=None
        if isinstance(MAX_REVIEWS_SCAN_BY_CATEGORY,dict): #dipende se max reviews è dizionario prende limite per categoria
            max_scan=MAX_REVIEWS_SCAN_BY_CATEGORY.get(label)
        max_saved=None
        if isinstance(MAX_RATINGS_SAVED_BY_CATEGORY,dict):
            max_saved=MAX_RATINGS_SAVED_BY_CATEGORY.get(label)
        #scansioniamo tutte le reviews
        review_stream=load_dataset( #carichi review della categoria corrente
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_review_{hf_category}",
            split="full",
            streaming=True, #legge righe progressivamente
            trust_remote_code=True,
        )

        matched_ratings=0
        scanned=0

        for row in tqdm(review_stream,desc=f"Filtro rating {label}"): #legge una review alla volta
            if max_saved is not None and matched_ratings>=max_saved:
                break
            if max_scan is not None and scanned>=max_scan: #se raggiunto limite, interrompe
                break
            scanned+=1
            parent_asin=row.get("parent_asin")
            if parent_asin not in category_items: #skippa reviews di prodotti non nel catalogo finale
                continue

            out_row={ #riga rating finale
                "user_id":row.get("user_id"),
                "parent_asin":parent_asin,
                "source_category":label,
                "hf_category":hf_category,
                "qa_slug":category["qa_slug"],
                "category_group":category["group"],
                "asin":row.get("asin"),
                "rating":tofloat_or_none(row.get("rating")),
                "timestamp":toint_or_none(row.get("timestamp")),
                "verified_purchase":row.get("verified_purchase"),
            }
            if SAVE_TEXT_OFREVIEW: #piu testo se lo vogliamo, per ora no, per alleggerire e testare modello base
                out_row["review_title"]=row.get("title")
                out_row["review_text"]=row.get("text")

            buffer.append(out_row)
            matched_ratings+=1
            matched_ratings_total+=1

            if len(buffer)>=200_000: #buffer pieno,scrivo su ratings.parquet
                writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
                buffer=[]

        scanned_total+=scanned
        for row_summary in category_summary_rows:
            if row_summary["source_category"]==label:
                row_summary["reviews_scanned"] = int(scanned)
                row_summary["ratings_saved"] = int(matched_ratings)
                break
        print(label,"| review scansionate:",scanned,"| rating salvati:",matched_ratings)

    writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
    if writer is not None:
        writer.close()

    print("Review scansionate totali:",scanned_total)
    print("Rating salvati totali:",matched_ratings_total)
    print("Output rating:",RATINGS_OUT)

Filtro rating Electronics: 1200000it [00:47, 25343.97it/s]


Electronics | review scansionate: 1200000 | rating salvati: 30249


Filtro rating Video Games: 400000it [00:16, 23917.65it/s]


Video Games | review scansionate: 400000 | rating salvati: 24128


Filtro rating Software: 250000it [00:14, 17360.09it/s]


Software | review scansionate: 250000 | rating salvati: 4809


... (22 outputs left out)

Filtro rating Tools and Home Improvement: 300000it [00:17, 17233.23it/s]


Tools and Home Improvement | review scansionate: 300000 | rating salvati: 1317


Filtro rating Toys and Games: 300000it [00:14, 21023.08it/s]


Toys and Games | review scansionate: 300000 | rating salvati: 3699
Review scansionate totali: 6050000
Rating salvati totali: 125029
Output rating: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset3_intersection\ratings.parquet


In [11]:
#allineamento items e qa ai soli prodotti che hanno rating
#così il dataset finale contiene prodotti con metadata + QA + rating
ratings_df=pd.read_parquet(RATINGS_OUT,columns=["parent_asin","source_category"]) #carica solo colonne utili
rating_products=set(ratings_df["parent_asin"]) #insieme di prodotti che hanno almeno un rating

items_df=items_df[ #filtra items df tenendo solo quelli presenti in ratings products
    items_df["parent_asin"].isin(rating_products)
].copy()
items_df.to_parquet(ITEMS_OUT,index=False)
items_id=set(items_df["parent_asin"])

valid_products=set(items_df["parent_asin"]) #filtra anche qadf
qa_df=qa_df[
    qa_df["parent_asin"].isin(valid_products)
].copy()
qa_df.to_parquet(QA_OUT,index=False)

category_final_counts=( #conta prodotti finali per categoria
    items_df.groupby("source_category")["parent_asin"]
    .nunique()
    .rename("final_products_qa_metadata_rating")
    .reset_index()
)

for row_summary in category_summary_rows: #aggiorna riepilogo per categoria, aggiungendo num prodotti finali
    label=row_summary["source_category"]
    final_count=category_final_counts.loc[
        category_final_counts["source_category"]==label,
        "final_products_qa_metadata_rating"
    ]
    row_summary["final_products_qa_metadata_rating"]=int(final_count.iloc[0]) if len(final_count) else 0

category_summary_df=pd.DataFrame(category_summary_rows)
category_summary_df.to_csv(CATEGORY_SUMMARY_PATH,index=False)

print("Prodotti finali con metadata + QA + rating:",len(items_df))
print("Prodotti con QA dopo aver filtrato:",qa_df["parent_asin"].nunique())
print("Righe QA dopo filtro:",len(qa_df))
print("Riepilogo per categoria:",CATEGORY_SUMMARY_PATH)
display(category_summary_df)

Prodotti finali con metadata + QA + rating: 11451
Prodotti con QA dopo aver filtrato: 11451
Righe QA dopo filtro: 94100
Riepilogo per categoria: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\test_dataset3_intersection\category_summary.csv


,source_category,hf_category,qa_slug,category_group,qa_rows,qa_products,metadata_rows,items_qa_metadata,duplicate_items_skipped,qa_rows_after_metadata,qa_products_after_metadata,reviews_scanned,ratings_saved,final_products_qa_metadata_rating
0,Electronics,Electronics,Electronics,tech,314263,39371,400000,6142,0,48316,6142,1200000,30249,4153
1,Video Games,Video_Games,Video_Games,tech,13307,1188,116964,600,0,6536,600,400000,24128,580
2,Software,Software,Software,tech,10636,1191,85068,600,0,5371,600,250000,4809,489
3,Office Products,Office_Products,Office_Products,office,43608,5681,357577,1500,0,11441,1500,600000,14912,1189
4,Appliances,Appliances,Appliances,home,9011,1153,78147,600,0,4619,600,250000,13323,533
5,Musical Instruments,Musical_Instruments,Musical_Instruments,music,23322,3002,136734,1200,0,8921,1200,400000,14078,1067
6,Arts Crafts and Sewing,Arts_Crafts_and_Sewing,Arts_Crafts_and_Sewing,crafts,21262,2944,488101,1000,0,7181,1000,500000,7554,785
7,Automotive,Automotive,Automotive,automotive,89923,12536,106545,500,0,3459,500,250000,690,234
8,Baby Products,Baby_Products,Baby,baby,28933,3783,59199,500,0,3882,500,250000,4295,363
9,Grocery and Gourmet Food,Grocery_and_Gourmet_Food,Grocery_and_Gourmet_Food,grocery,19538,3017,150000,333,0,2162,333,250000,868,200


In [12]:
def parquet_count_rows(path:Path)->int: #funzione che conta righe di un file parquet
    return pq.ParquetFile(path).metadata.num_rows

summary={ #dizionario riassuntivo
    "dataset_name":DATASET_NAME,
    "dataset_label":DATASET_LABEL,
    "categories":[category["label"] for category in CATEGORIES],
    "items":parquet_count_rows(ITEMS_OUT),
    "qa":parquet_count_rows(QA_OUT),
    "ratings":parquet_count_rows(RATINGS_OUT),
    "category_summary_path":str(CATEGORY_SUMMARY_PATH),
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
summary

{'dataset_name': 'test_dataset3',
 'dataset_label': 'AmazonLike_v2',
 'categories': ['Electronics',
  'Video Games',
  'Software',
  'Office Products',
  'Appliances',
  'Musical Instruments',
  'Arts Crafts and Sewing',
  'Automotive',
  'Baby Products',
  'Grocery and Gourmet Food',
  'Home and Kitchen',
  'Patio Lawn and Garden',
  'Pet Supplies',
  'Sports and Outdoors',
  'Tools and Home Improvement',
  'Toys and Games'],
 'items': 11451,
 'qa': 94100,
 'ratings': 125029,
 'category_summary_path': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\data\\processed\\test_dataset3_intersection\\category_summary.csv'}